In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense
from sklearn.preprocessing import normalize
from sklearn.metrics import accuracy_score, confusion_matrix

# ======================
# 1. Data Loader (AUTO SPLIT)
# ======================
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,        # 80% train, 20% test
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

train_gen = datagen.flow_from_directory(
    'SkinDiseaseClass',
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

test_gen = datagen.flow_from_directory(
    'SkinDiseaseClass',
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# ======================
# 2. CNN Gaussian Embedding
# ======================
base = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')

x = base.output
x = Dense(128, activation='relu')(x)
x = Dense(64, activation='relu')(x)
out = Dense(5, activation='softmax')(x)

classifier = Model(base.input, out)

for layer in base.layers:
    layer.trainable = False

# ======================
# 3. Train CNN head
# ======================
classifier.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

classifier.fit(train_gen, epochs=12)

# ======================
# 4. Feature Extract
# ======================
feature_extractor = Model(classifier.input, classifier.layers[-2].output)

X_train = feature_extractor.predict(train_gen)
X_test  = feature_extractor.predict(test_gen)
y_train = train_gen.classes
y_test  = test_gen.classes

# ======================
# 5. Normalize
# ======================
X_train = normalize(X_train)
X_test  = normalize(X_test)

# ======================
# 6. Bayes Decision Theory
# ======================
class GaussianBayes:
    def __init__(self):
        self.mean = {}
        self.var  = {}
        self.prior = {}

    def fit(self, X, y):
        for c in np.unique(y):
            Xc = X[y == c]
            self.mean[c] = Xc.mean(axis=0)
            self.var[c]  = Xc.var(axis=0) + 0.01
            self.prior[c] = len(Xc) / len(X)

    def predict(self, X):
        preds = []
        for x in X:
            scores = {}
            for c in self.mean:
                loglik = -0.5 * np.sum(
                    np.log(2*np.pi*self.var[c]) +
                    ((x - self.mean[c])**2) / self.var[c]
                )
                scores[c] = loglik + np.log(self.prior[c])
            preds.append(max(scores, key=scores.get))
        return np.array(preds)

# ======================
# 7. Train Bayes & Evaluate
# ======================
model = GaussianBayes()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))


Found 6801 images belonging to 5 classes.
Found 1698 images belonging to 5 classes.


C:\Users\HP\AppData\Local\Temp\ipykernel_19500\3894019157.py:41: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')


Epoch 1/12
213/213 ━━━━━━━━━━━━━━━━━━━━ 368s 2s/step - accuracy: 0.6745 - loss: 0.8999
Epoch 2/12
213/213 ━━━━━━━━━━━━━━━━━━━━ 211s 987ms/step - accuracy: 0.8586 - loss: 0.4242
Epoch 3/12
213/213 ━━━━━━━━━━━━━━━━━━━━ 202s 946ms/step - accuracy: 0.8971 - loss: 0.3044
Epoch 4/12
213/213 ━━━━━━━━━━━━━━━━━━━━ 204s 956ms/step - accuracy: 0.9213 - loss: 0.2448
Epoch 5/12
213/213 ━━━━━━━━━━━━━━━━━━━━ 182s 854ms/step - accuracy: 0.9319 - loss: 0.2048
Epoch 6/12
213/213 ━━━━━━━━━━━━━━━━━━━━ 192s 902ms/step - accuracy: 0.9425 - loss: 0.1764
Epoch 7/12
213/213 ━━━━━━━━━━━━━━━━━━━━ 271s 1s/step - accuracy: 0.9525 - loss: 0.1538
Epoch 8/12
213/213 ━━━━━━━━━━━━━━━━━━━━ 238s 1s/step - accuracy: 0.9560 - loss: 0.1359
Epoch 9/12
213/213 ━━━━━━━━━━━━━━━━━━━━ 189s 887ms/step - accuracy: 0.9650 - loss: 0.1175
Epoch 10/12
213/213 ━━━━━━━━━━━━━━━━━━━━ 193s 905ms/step - accuracy: 0.9677 - loss: 0.1066
Epoch 11/12
213/213 ━━━━━━━━━━━━━━━━━━━━ 191s 895ms/step - accuracy: 0.9719 - loss: 0.0914
Epoch 12/12
213/2

In [4]:
import joblib

joblib.dump(model, "bayes_skin_model.pkl")
joblib.dump(train_gen.class_indices, "class_names.pkl")
feature_extractor.save("cnn_feature_model.h5")